# Going Further with JAX — Fourier Neural Operators (Student)

 **Optional self-paced extension — NOT part of the two mandatory 45-minute practicals.**

 **Who is it for?** Physical scientists, ML practitioners, and newcomers. This extension reproduces one idea from the PyTorch lab in **JAX** and explains what changes. No prior JAX experience is assumed.

 **By the end, you can:** work with immutable arrays and PRNG keys; explain `jax.jit`, automatic differentiation and asynchronous GPU execution; write a Fourier spectral operator; distinguish compilation time from steady-state step time.

 **Installation:** JAX, Equinox and Optax must already be available in the remote Python environment. The notebook runs on CPU or GPU. Avoid installing during the 90-minute session. The numerical example is synthetic and educational, **not** a geophysical accuracy benchmark.

**Hedy, one-terminal Euporie + X11:** From a single SSH/X11 connection to `m-login01`, run `bash scripts/start_lab.sh fno student` (or `gnn student`, `infrasound student`, `ffno_map`, `jax_fno student`, `jax_gnn student`). The launcher submits a Slurm GPU allocation, connects to its compute node with SSH X11, enters a Slurm job step and opens this notebook in Euporie. Figures use TkAgg windows via X11; no web browser or local Jupyter is needed. Student edits are saved as private notebook copies. See `README_ONE_TERMINAL_X11.md`.

**Course timing:** mandatory FNO and GNN labs are 45 minutes each; all extensions are optional. **GPU type:** `HEDY_GPU_PROFILE=full` uses an H100; `mig` requests `1g.20gb` when available. Ensure an active site-authorized reservation.

## 0. Device and numerical habits

 JAX arrays are immutable: use `.at[idx].set(...)` rather than in-place assignment. A `jax.random.PRNGKey` is explicit and is split for reproducible sampling. `jax.jit` traces/compiles functions for a fixed shape and dtype; the first call includes compilation. **On GPU, synchronize** with `.block_until_ready()` before timing.

 Equinox (`eqx.Module`) stores trainable arrays in PyTrees; Optax provides optimizer state and updates. Do not assume that a visible device name means you have a valid Slurm GPU allocation.

---
**Authors:** E. Noëlé¹²³⁴, J.-P. Gatti⁵, L. Lehmann⁵, C. Millet⁵, D. Clouteau³

**Affiliations:**
¹ AMIAD — ² CEA — ³ LMPS (Laboratoire de Mécanique Paris-Saclay) — ⁴ Centre Borelli — ⁵ CEA/DAM

**LETMA 2026 Summer School** · Machine Learning on Geophysical Data


In [ ]:
import os, time
import numpy as np
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'tk' if os.getenv('LAB_PLOT_MODE', 'inline').lower() == 'tk' else 'inline')
import matplotlib.pyplot as plt

# Faster X11 preview: lower DISPLAY DPI, preserve high-resolution file exports.

if os.environ.get('LAB_PLOT_MODE', 'inline').lower() == 'tk' and os.environ.get('HEDY_FAST_FIGURES', '1') == '1':
    plt.rcParams.update({
        'figure.dpi': 85,
        'savefig.dpi': 200,
        'path.simplify': True,
        'agg.path.chunksize': 12000,
    })

# Choose a non-interactive backend for profiling; on Hedy X11 use LAB_PLOT_MODE=tk.
print("JAX", jax.__version__, "| devices", jax.devices())
print("Platform:", jax.default_backend(), "| SLURM_JOB_ID:", os.getenv("SLURM_JOB_ID"))
key = jax.random.PRNGKey(42)

In [ ]:
from hedy_euporie.profiler import begin_session
HEDY_SESSION = begin_session('jax_fno_student', backend='jax')

## 1. Physics: diffusion as a Fourier multiplier

 For a periodic scalar field, $\partial_t u=\nu\partial_{xx}u$ has Fourier solution $\widehat u_k(t)=e^{-\nu(2\pi k)^2t}\widehat u_k(0)$. The damping is a **known** physical multiplier. A trainable spectral layer replaces it with freely learned complex coefficients; it is **not automatically a diffusion solver**. We use smooth periodic random initial fields so the exercise is small and reproducible.

In [ ]:
N = 64
x_coord = jnp.arange(N, dtype=jnp.float32) / N
K = 8
key, key_a, key_phase = jax.random.split(key, 3)
amplitude = jax.random.normal(key_a, (96, K), dtype=jnp.float32) / jnp.arange(1, K+1)[None, :]**2
phase = jax.random.uniform(key_phase, (96, K), minval=0., maxval=2*jnp.pi)
freq = jnp.arange(1, K+1, dtype=jnp.float32)
x_data = jnp.sum(amplitude[:, :, None] * jnp.sin(2*jnp.pi*freq[None, :, None]*x_coord[None, None, :] + phase[:,:,None]), axis=1)
nu_dt = 0.003
wave_numbers = jnp.fft.rfftfreq(N, d=1/N)
heat_multiplier = jnp.exp(-nu_dt*(2*jnp.pi*wave_numbers)**2)
y_data = jnp.fft.irfft(jnp.fft.rfft(x_data, axis=-1)*heat_multiplier[None,:], n=N, axis=-1)
x_train, x_test = x_data[:80], x_data[80:]
y_train, y_test = y_data[:80], y_data[80:]
print('Shapes:', x_train.shape, y_train.shape, '| dtype:', x_train.dtype)
fig, ax = plt.subplots(figsize=(8,3)); ax.plot(np.asarray(x_coord),np.asarray(x_test[0]),label='Initial'); ax.plot(np.asarray(x_coord),np.asarray(y_test[0]),label='Diffused'); ax.set(xlabel='Position',ylabel='Amplitude'); ax.legend(); ax.grid(alpha=.3); plt.show()

## 2. Exercise JF1 — implement a learned spectral layer

 Input: `x` with shape `(batch, N)`. We retain the first `modes` non-negative frequencies and learn a **complex** multiplier. Use `rfft` and `irfft(n=N)`, allocating the output spectrum with `zeros_like`. **Reasoning question:** what happens to the high modes discarded by this layer?

 Student scaffold falls back to a correct reference implementation until you replace it, so later cells can still run. Check both `shape` and numerical agreement.

In [ ]:
def reference_spectral_apply(x, w):
    spectrum = jnp.fft.rfft(x, axis=-1)
    out = jnp.zeros_like(spectrum)
    m = min(w.shape[0], spectrum.shape[-1])
    out = out.at[:, :m].set(spectrum[:, :m]*w[:m][None,:])
    return jnp.fft.irfft(out, n=x.shape[-1], axis=-1)

def spectral_apply(x, w):
    # TODO JF1: take the rFFT; construct an output spectrum; multiply low modes;
    # inverse rFFT using n=x.shape[-1].
    return reference_spectral_apply(x, w)  # safe fallback, replace with your implementation

w_probe = jnp.ones((12,), dtype=jnp.complex64)
test_out = spectral_apply(x_train[:3], w_probe)
np.testing.assert_allclose(np.asarray(test_out), np.asarray(reference_spectral_apply(x_train[:3],w_probe)),atol=1e-5)
assert test_out.shape == x_train[:3].shape
print('JF1 numerical and shape checks: PASS')

## 3. Equinox + Optax: learn a spectral operator

 Here trainable real and imaginary arrays form a complex multiplier. `eqx.filter_value_and_grad` differentiates floating-point trainable arrays, while `eqx.filter_jit` compiles the update. We only fit the low-frequency band; the target diffusion dynamics are completely specified by the analytic solution above.

 **Exercise JF2:** explain why the gradient of the **mean squared error** is needed and why optimizer state must be carried between updates.

In [ ]:
class SpectralOperator(eqx.Module):
    real: jax.Array
    imag: jax.Array
    def __init__(self, key, modes=12):
        self.real = jnp.ones((modes,), dtype=jnp.float32)
        self.imag = jnp.zeros((modes,), dtype=jnp.float32)
    def __call__(self, x):
        return spectral_apply(x, self.real + 1j*self.imag)

model = SpectralOperator(key)
optimizer = optax.adam(learning_rate=0.04)
opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))

def mse(model, x, y):
    return jnp.mean((model(x) - y)**2)

@eqx.filter_jit
def step(model, opt_state, x, y):
    loss, grads = eqx.filter_value_and_grad(mse)(model, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_inexact_array))
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

t0=time.perf_counter()
model, opt_state, first_loss = step(model, opt_state, x_train, y_train)
first_loss.block_until_ready()
compile_plus_step_s = time.perf_counter()-t0
for i in range(80):
    model, opt_state, train_loss = step(model, opt_state, x_train, y_train)
train_loss.block_until_ready()
print(f'First JIT call (includes compilation): {compile_plus_step_s:.3f}s')
print(f'Train MSE: {float(train_loss):.6g} | Test MSE: {float(mse(model,x_test,y_test)):.6g}')
fig, ax=plt.subplots(figsize=(8,3)); ax.plot(np.asarray(x_coord),np.asarray(y_test[0]),label='Target'); ax.plot(np.asarray(x_coord),np.asarray(model(x_test[:1])[0]),'--',label='JAX'); ax.set(xlabel='x',ylabel='u'); ax.grid(alpha=.3);ax.legend();plt.show()

## 4. Timing and interpretation (optional)

 Separate compilation from execution. Benchmark on an **allocated** device and synchronize. GPU profiling must also account for memory reservation, launch overhead and the batching used; this toy example is not a meaningful GPU-throughput benchmark.

In [ ]:
t0=time.perf_counter()
_, _, loss = step(model, opt_state, x_train, y_train)
loss.block_until_ready()
print(f'Compiled training step (synchronized): {time.perf_counter()-t0:.6f} s')
print('Learned complex modes:', np.asarray(model.real[:4] + 1j*model.imag[:4]))

**Takeaways:** rFFT encodes nonlocal interactions on a regular grid; learning spectral weights does not impose a PDE; JAX state is explicit and updates are functional; JIT compilation must be excluded from steady-state timings. Going further: compare the learned multiplier to analytic heat damping and inspect which frequencies are absent in the training data.

### Optional — save this interactive session measurement

Run the next cell after finishing your exercise. It writes a JSON report with elapsed **interactive** time (including time spent reading and editing), maximum Python-kernel RSS, and (for PyTorch) peak CUDA memory. It does **not** measure Slurm billed time or prove that all cells were executed.

In [ ]:
HEDY_SESSION.report()

## References

- Z. Li et al. *Fourier Neural Operator for Parametric PDEs.* ICLR 2021.
- Z. Li et al. *Neural Operator: Learning Maps Between Function Spaces.* Nature MI **6**, 666 (2024).
- A. Paszke et al. *PyTorch.* NeurIPS 2019.
- J. Bradbury et al. *JAX: Composable Transformations of Python+NumPy Programs* (2018).
- A. Paszke et al. *Equinox: Your Friend for JAX* (2022).
- J. O. Richardson et al. *Optax: Composable Gradient Processing and Optimization* (2023).
